## 子图

### 子图调用

子图（Subgraph）是编译后的图，可以像普通节点一样被父图复用。按父子状态结构是否兼容，有两种调用方式：

- **状态结构不同**：在节点函数里手动调用 `subgraph.invoke()`，自己做输入 / 输出映射。
- **共享状态键**：直接把编译后的子图传给 `add_node`，LangGraph 自动完成状态传递。

#### 方式一、在节点函数中调用子图（状态结构不同）

当父子状态没有共享键时，不能把子图作为节点添加，需要在节点函数里手动 `invoke`，并显式完成输入 / 输出字段的映射。此时子图只是节点内部的一次普通执行，不会出现在父图的结构图里。

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


# ===== 子图 =====
class TranslateState(TypedDict):
    text: str
    result: str

def translate(state: TranslateState):
    return {"result": f"翻译结果：{state['text']}"}

sub_builder = StateGraph(TranslateState)
sub_builder.add_node("translate", translate)
sub_builder.add_edge(START, "translate")
sub_builder.add_edge("translate", END)
translate_subgraph = sub_builder.compile()


# ===== 父图 =====
class ParentState(TypedDict):
    question: str
    answer: str

def call_subgraph(state: ParentState):
    subgraph_output = translate_subgraph.invoke({"text": state["question"]})
    return {"answer": subgraph_output["result"]}

builder = StateGraph(ParentState)
builder.add_node("call_subgraph", call_subgraph)
builder.add_edge(START, "call_subgraph")
builder.add_edge("call_subgraph", END)
graph = builder.compile()

result = graph.invoke({"question": "你好"})
print(result)

In [ ]:
from IPython.display import display, Image

# 只显示父图
display(graph)

# 显示父子图（底层识别到对子图的调用）
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

#### 方式二、把子图作为节点添加（共享状态键）

父子状态至少有一个同名键即可通信；子图独有的键只在子图内部生效，返回父图时会被丢弃。

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


# ===== 子图：文本清洗 + 润色 =====
class SubgraphState(TypedDict):
    raw_text: str
    clean_text: str
    final_text: str

def clean(state: SubgraphState):
    return {"clean_text": state["raw_text"].strip()}

def polish(state: SubgraphState):
    return {"final_text": f"{state['clean_text']}（已润色）"}


sub_builder = StateGraph(SubgraphState)
sub_builder.add_node("clean", clean)
sub_builder.add_node("polish", polish)
sub_builder.add_edge(START, "clean")
sub_builder.add_edge("clean", "polish")
sub_builder.add_edge("polish", END)
subgraph = sub_builder.compile()


# ===== 父图 =====
class ParentState(TypedDict):
    raw_text: str
    final_text: str


builder = StateGraph(ParentState)
builder.add_node("subgraph_node", subgraph)
builder.add_edge(START, "subgraph_node")
builder.add_edge("subgraph_node", END)
graph = builder.compile()

result = graph.invoke({"raw_text": "  LangGraph 子图示例  "})
print(result)

In [ ]:
from IPython.display import display, Image

display(Image(graph.get_graph(xray=True).draw_mermaid_png()))